In [1]:
import os
import pandas as pd
import plotly.graph_objects as go
from typing import List, Optional, Dict
import numpy as np

def find_config_file(folder_path: str) -> Optional[str]:
    """
    Finds a configuration file (ending with .txt) within the 'configs' subfolder.
    """
    config_dir = os.path.join(folder_path, 'configs')
    if not os.path.isdir(config_dir):
        return None
    for item in os.listdir(config_dir):
        if item.endswith('.txt'):
            return os.path.join(config_dir, item)
    return None

def parse_config(file_path: str) -> Dict[str, str]:
    """
    Parses a 'key = value', 'key value', or 'key: value' configuration file into a dictionary.
    """
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                
                separator = None
                if ':' in line:
                    separator = ':'
                elif '=' in line:
                    separator = '='

                if separator:
                    parts = line.split(separator, 1)
                else:
                    parts = line.split(None, 1)

                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except FileNotFoundError:
        print(f"Config file not found: {file_path}")
    except Exception as e:
        print(f"Error parsing config file {file_path}: {e}")
    return params

def find_timing_file(run_folder_path: str, sim_type: str) -> Optional[str]:
    """Finds the ...trace_matched_timing.csv file for a given run."""
    sim_output_dir = os.path.join(run_folder_path, sim_type.lower())
    if not os.path.isdir(sim_output_dir):
        return None
    for f in os.listdir(sim_output_dir):
        if 'trace_matched_timing.csv' in f:
            return os.path.join(sim_output_dir, f)
    return None

# --- Collective Info Parsing ---
def parse_collectives_log(log_path: str) -> Dict[str, Dict]:
    """Parses a duplicate_collectives.log file to extract signatures."""
    collective_info = {}
    try:
        with open(log_path, 'r') as f:
            lines = f.readlines()
            i = 0
            while i < len(lines):
                line = lines[i]
                workload_match = re.match(r'^Workload: (\S+)', line)
                if workload_match:
                    current_workload = workload_match.group(1)
                    # Look for signature on the next line
                    if (i + 1 < len(lines)) and (signature_match := re.match(r'^\s+Signature: \((.*)\)', lines[i+1])):
                        sig_content = signature_match.group(1).strip()
                        # Split signature into its three parts
                        parts = sig_content.rsplit(', ', 2)
                        if len(parts) == 3:
                            npu_tuples, comm_type, comm_size = parts
                            collective_info[current_workload] = {
                                'npu_tuples': npu_tuples.strip(),
                                'comm_type': comm_type.strip(),
                                'comm_size': comm_size.strip()
                            }
                i += 1
    except FileNotFoundError:
        print(f"Warning: Collectives log file not found at {log_path}")
    except Exception as e:
        print(f"Error parsing collectives log {log_path}: {e}")
    return collective_info

In [2]:
import re
import pandas as pd
from plotly.subplots import make_subplots
# --- Configuration for G2 vs NS3 Comparison ---

# Define the base folder containing all the run directories from all workloads
# We will process all workloads ('toy_all_to_all_one_collective', 'toy_all_reduce_one_collective', 'model')
# base_comparison_folder = '/app/astra-sim /upc/output/comparison_run/FoldedClos'
# base_comparison_folder = '/app/astra-sim/upc/output/comparison_run/FoldedClos/basic_model_0_split'
# /app/astra-sim/upc/output/comparison_run/experiment1/FoldedClos/multiple_collectives
base_comparison_folders = [
    '/app/astra-sim/upc/output/comparison_run/experiment1/FoldedClos/multiple_collectives']

# Choose what to plot: 'avg' for mean, or 'max' for maximum value
comparison_plot_metric = 'max'  # Can be 'avg' or 'max'

# --- Data Collection Logic ---

all_run_folders = []
for base_comparison_folder in base_comparison_folders:
    for workload_folder in os.listdir(base_comparison_folder):
        workload_path = os.path.join(base_comparison_folder, workload_folder)
        if os.path.isdir(workload_path):
            run_folders = [os.path.join(workload_path, d) for d in os.listdir(workload_path) if os.path.isdir(os.path.join(workload_path, d))]
            all_run_folders.extend(run_folders)

comparison_results = []

# Regex to extract topology index
topo_idx_regex = re.compile(r'_v(\d+)')

cc_modes = {0: "PFC", 1: "DCQCN", 3: "HPCC", 7: "TIMELY", 8: "DCTCP", 10: "HPCC-PINT"}

def parse_runtime(time_str: str) -> float:
    """Parses 'H:MM:SS.ffffff' into seconds."""
    if not time_str:
        return 0.0
    try:
        parts = time_str.split(':')
        h = int(parts[0])
        m = int(parts[1])
        s = float(parts[2])
        return h * 3600 + m * 60 + s
    except (ValueError, IndexError):
        return 0.0


for folder in sorted(all_run_folders):
    run_summary_path = os.path.join(folder, 'run_summary.txt')
    if not os.path.exists(run_summary_path):
        continue

    # 1. Parse run_summary.txt to identify sim_type and topology
    summary_params = parse_config(run_summary_path)

    workload_name = summary_params.get('collective', 'N/A').strip()
    npu_count = summary_params.get('npus count', 'N/A')
    total_runtime_str = summary_params.get('total runtime', '0:0:0.0')
    execution_time_sec = parse_runtime(total_runtime_str)


    # Process 'analytical_unaware' separately as it's topology-independent
    if os.path.exists(os.path.join(folder, 'analytical_unaware')):
        timing_file = None
        sim_output_dir = os.path.join(folder, 'analytical_unaware')
        for f in os.listdir(sim_output_dir):
            if 'trace_matched_timing.csv' in f:
                timing_file = os.path.join(sim_output_dir, f)
                break

        if timing_file:
            try:
                df = pd.read_csv(timing_file)
                time_col = 'callback_tick'
                if time_col in df.columns:
                    elapsed_times = df[time_col].dropna()
                    if not elapsed_times.empty:
                        comparison_results.append({
                            'workload': workload_name,
                            'npu_count': npu_count,
                            'topo_index': -1,  # Use -1 to indicate topology independence
                            'sim_type': 'Analytical Unaware',
                            'run_name': 'Analytical Unaware',
                            'avg_time': elapsed_times.mean(),
                            'max_time': elapsed_times.max(),
                            'std_dev': elapsed_times.std(),
                            'execution_time': execution_time_sec,
                            'path': folder
                        })
            except Exception as e:
                print(f"Error processing analytical_unaware in {folder}: {e}")

    # Process topology-dependent simulations (G2, NS3)
    sim_type = None
    topo_file = None
    if os.path.exists(os.path.join(folder, 'g2')):
        sim_type = 'G2'
        topo_file = summary_params.get('g2 topology file override', 'N/A')
    elif os.path.exists(os.path.join(folder, 'ns3')):
        sim_type = 'NS3'
        topo_file = summary_params.get('ns3 topology file override', 'N/A')

    if not sim_type or not topo_file or 'all_paths' in topo_file:
        continue

    # Extract topology index
    match = topo_idx_regex.search(topo_file)
    if not match:
        continue
    topo_index = int(match.group(1))

    # 2. Get timing data
    timing_file = None
    sim_output_dir = None
    if sim_type == 'G2':
        sim_output_dir = os.path.join(folder, 'g2')
    elif sim_type == 'NS3':
        sim_output_dir = os.path.join(folder, 'ns3')

    if sim_output_dir and os.path.isdir(sim_output_dir):
        for f in os.listdir(sim_output_dir):
            if 'trace_matched_timing.csv' in f:
                timing_file = os.path.join(sim_output_dir, f)
                break

    if not timing_file:
        continue

    try:
        df = pd.read_csv(timing_file)
        if sim_type == 'NS3':
            df = df[df['node_name'] != 'dummy_node'].copy()

        time_col = 'callback_tick'
        if time_col not in df.columns:
            continue

        elapsed_times = df[time_col].dropna()
        if elapsed_times.empty:
            continue

        # 3. Create a descriptive name and store results
        run_name = f"{sim_type}"
        if sim_type == 'NS3':
            ns3_config_file = find_config_file(folder)
            if ns3_config_file:
                ns3_params = parse_config(ns3_config_file)
                cc_mode_val = int(ns3_params.get('cc_mode', -1))
                cc_name = cc_modes.get(cc_mode_val, 'N/A')
                
                # Get distinguishing parameters
                packet_payload = ns3_params.get('packet_payload_size', 'N/A')
                buffer_size = ns3_params.get('buffer_size', 'N/A')
                has_win = ns3_params.get('has_win', 'N/A')
                var_win = ns3_params.get('var_win', 'N/A')
                tinc = ns3_params.get('rp_timer', 'N/A')
                
                # Extract kmax and kmin values from the map strings
                kmax_map = ns3_params.get('kmax_map', 'N/A')
                kmin_map = ns3_params.get('kmin_map', 'N/A')
                kmax_val = kmax_map.split()[2] if kmax_map != 'N/A' and len(kmax_map.split()) > 2 else 'N/A'
                kmin_val = kmin_map.split()[2] if kmin_map != 'N/A' and len(kmin_map.split()) > 2 else 'N/A'

                # Distinguish DCQCN variants (8 total: 2 tinc × 2 k_base × 2 packet_payload)
                if cc_name == 'DCQCN':
                    if tinc == '300':
                        cc_name = 'DCQCN1'
                    elif tinc == '55':
                        cc_name = 'DCQCN2'
                    # Add kmax/kmin and packet payload to distinguish all 8 variants
                    run_name = f"{cc_name}_k{kmax_val}-{kmin_val}_pkt{packet_payload}"
                
                # Distinguish TIMELY variants (4 total: 2 var_win × 2 packet_payload)
                elif cc_name == 'TIMELY':
                    if var_win == '1':
                        cc_name = 'TIMELY_vwin'
                    else:
                        cc_name = 'TIMELY'
                    run_name = f"{cc_name}_pkt{packet_payload}"
                
                # For other protocols, include packet payload
                else:
                    run_name = f"{cc_name}_pkt{packet_payload}"

        comparison_results.append({
            'workload': workload_name,
            'npu_count': npu_count,
            'topo_index': topo_index,
            'sim_type': sim_type,
            'run_name': run_name,
            'avg_time': elapsed_times.mean(),
            'max_time': elapsed_times.max(),
            'min_time': elapsed_times.min(),
            'std_dev': elapsed_times.std(),
            'execution_time': execution_time_sec,
            'path': folder
        })

    except Exception as e:
        print(f"Error processing {folder}: {e}")

# --- Plotting Logic ---

if comparison_results:
    comp_df = pd.DataFrame(comparison_results)

    # Create a directory to store the plots
    plot_output_folder = '/app/astra-sim/upc/comparing_networks/comparison_plots/'
    os.makedirs(plot_output_folder, exist_ok=True)

    # Determine which column to use for plotting
    if comparison_plot_metric == 'max':
        y_col = 'max_time'
        y_axis_title = "Estimated Execution Time (ns)"
    elif comparison_plot_metric == 'min':
        y_col = 'min_time'
        y_axis_title = "Minimum Time (ns)"
    else: # Default to 'avg'
        y_col = 'avg_time'
        y_axis_title = "Average Time (ns)"

    workloads = comp_df['workload'].unique()

    for wl in sorted(workloads):
        workload_df = comp_df[comp_df['workload'] == wl]

        # For this workload, find the single analytical unaware time, if it exists.
        au_runs = workload_df[workload_df['sim_type'] == 'Analytical Unaware']
        au_time = None
        if not au_runs.empty:
            au_time = au_runs.iloc[0][y_col]

        for topo_idx in sorted(workload_df['topo_index'].unique()):
            # Skip the placeholder index used for analytical_unaware
            if topo_idx == -1:
                continue

            group_df = workload_df[workload_df['topo_index'] == topo_idx].copy()

            if group_df.empty:
                continue

            # Separate G2 and NS3 for plotting
            g2_runs = group_df[group_df['sim_type'] == 'G2']
            ns3_runs = group_df[group_df['sim_type'] == 'NS3'].sort_values(by=y_col)

            if ns3_runs.empty:
                continue # Don't plot if there's no NS3 data to compare against

            # plot_title = f'G2 vs NS3 Comparison for Workload: "{wl}", Static routing: {topo_idx}'

            fig = go.Figure()

            # Add NS3 runs as bars to the first subplot
            fig.add_trace(go.Bar(
                x=ns3_runs['run_name'],
                y=ns3_runs[y_col],
                name='NS3 Runs',
                marker_color='rgb(55, 83, 109)',
                text=ns3_runs[y_col].apply(lambda x: f'{x/1e9:.4f} s'),
                textposition='outside'
            ))

            # --- Speedup Calculation & Annotation ---
            fastest_ns3_run = ns3_runs.iloc[0]
            fastest_ns3_time = fastest_ns3_run[y_col]
            npu_count_val = group_df['npu_count'].iloc[0] if not group_df.empty else 'N/A'

            g2_time = None
            g2_sim_time_error_text = "G2: N/A (No G2 run)"
            if not g2_runs.empty:
                g2_run = g2_runs.iloc[0]
                g2_time = g2_run[y_col]
                # Calculate percentage error relative to fastest NS3
                sim_time_error_pct = ((g2_time - fastest_ns3_time) / fastest_ns3_time) * 100
                g2_sim_time_error_text = f"G2: {sim_time_error_pct:+.2f}%"
                fig.add_hline(
                    y=g2_time,
                    line_dash="dot",
                    annotation_text=f"G2 Time: {g2_time/1e9:.4f} s",
                    annotation_position="top right",
                    line_color="red",
                    annotation=dict(font=dict(color="white", size=16), bgcolor="red", borderpad=4),
                )

            au_sim_time_error_text = "Unaware: N/A"
            if au_time is not None:
                # Calculate percentage error relative to fastest NS3
                sim_time_error_pct = ((au_time - fastest_ns3_time) / fastest_ns3_time) * 100
                au_sim_time_error_text = f"Unaware: {sim_time_error_pct:+.2f}%"
                fig.add_hline(
                    y=au_time,
                    line_dash="dash",
                    annotation_text=f"Analytical Unaware: {au_time/1e9:.4f} s",
                    annotation_position="bottom right",
                    line_color="green",
                    annotation=dict(font=dict(color="white", size=16), bgcolor="green", borderpad=4),
                )


            # --- Execution Time Speedup Calculation ---
            fastest_ns3_exec_time = ns3_runs['execution_time'].min()

            g2_exec_speedup_text = "N/A"
            if not g2_runs.empty:
                g2_exec_time = g2_runs.iloc[0]['execution_time']
                if g2_exec_time > 0:
                    exec_speedup = fastest_ns3_exec_time / g2_exec_time
                    g2_exec_speedup_text = f"{exec_speedup:.2f}x"

            au_exec_speedup_text = "N/A"
            if not au_runs.empty:
                au_exec_time = au_runs.iloc[0]['execution_time']
                if au_exec_time > 0:
                    exec_speedup = fastest_ns3_exec_time / au_exec_time
                    au_exec_speedup_text = f"{exec_speedup:.2f}x"

            # Construct the summary text
            summary_text = (
                f"<b>Summary</b><br>"
                f"--------------------<br>"
                f"<b>NPU Nodes:</b> {npu_count_val}<br>"
                f"<b>Static Routing:</b> {topo_idx}<br>"
                f"--------------------<br>"
                f"<b>Execution Time Error vs Fastest NS3 (%):</b><br>"
                f"- {g2_sim_time_error_text}<br>"
                f"- {au_sim_time_error_text}<br>"
                f"--------------------<br>"
                f"<b>Sim Time Speedup vs Fastest NS3:</b><br>"
                f"- G2: {g2_exec_speedup_text}<br>"
                # f"- Unaware: {au_exec_speecdup_text}"
            )

            fig.add_annotation(
                text=summary_text,
                align='left',
                showarrow=False,
                xref='paper',
                yref='paper',
                x=1.38,
                y=0.8,
                bordercolor="black",
                borderwidth=1,
                bgcolor="rgba(255, 255, 255, 0.8)",
                font=dict(size=16)
            )

            fig.update_layout(
                # title=plot_title,
                template='plotly_white',
                height=700,
                width=1600,
                margin=dict(b=40, r=450), # Increased right margin for the text box
                legend=dict(x=1.05, y=1.0),
                showlegend=False,
                font=dict(size=16)
            )

            # Update axes for both subplots
            fig.update_xaxes(title_text="RDMA Protocol", tickangle=-60, title_font=dict(size=18))
            fig.update_yaxes(title_text=y_axis_title, title_font=dict(size=18))

            fig.show()
            # Save the figure to a PDF file
            pdf_filename = f"topo{topo_idx}_{wl.replace('/', '_')}.pdf"
            pdf_filepath = os.path.join(plot_output_folder, pdf_filename)
            try:
                fig.write_image(pdf_filepath)
                print(f"Plot saved to {pdf_filepath}")
            except Exception as e:
                print(f"Error saving plot to {pdf_filepath}: {e}")
                print("Please ensure you have 'kaleido' installed (`pip install kaleido`) for PDF export.")
else:
    print("\nNo comparison results to plot.")

# Display the full data table
if comparison_results:
    print("\n--- Full Comparison Data ---")
    # Reorder columns for better readability
    display_cols = [
        'workload', 'topo_index', 'sim_type', 'run_name',
        y_col, 'std_dev', 'execution_time', 'npu_count',
        'npu_tuples', 'comm_type', 'comm_size', 'path'
    ]
    # Ensure all columns exist before trying to display them
    final_cols = [c for c in display_cols if c in comp_df.columns]
    with pd.option_context('display.max_rows', 10, 'display.max_columns', None, 'display.width', 1000):
        display(comp_df.sort_values(by=['workload', 'topo_index', y_col])[final_cols])


Plot saved to /app/astra-sim/upc/comparing_networks/comparison_plots/topo1_multiple_collectives_all_gather_size_134217728_group_0.pdf


Plot saved to /app/astra-sim/upc/comparing_networks/comparison_plots/topo1_multiple_collectives_all_gather_size_33554432_group_0.pdf

--- Full Comparison Data ---


,workload,topo_index,sim_type,run_name,max_time,std_dev,execution_time,npu_count,path
1,multiple_collectives/all_gather_size_134217728...,-1,Analytical Unaware,Analytical Unaware,5368729,0.000000e+00,1.230110,16,/app/astra-sim/upc/output/comparison_run/exper...
0,multiple_collectives/all_gather_size_134217728...,1,G2,G2,102413540,7.429009e+06,1.150811,16,/app/astra-sim/upc/output/comparison_run/exper...
3,multiple_collectives/all_gather_size_134217728...,1,NS3,DCTCP_pkt9000,103743105,4.585358e+06,506.859421,16,/app/astra-sim/upc/output/comparison_run/exper...
5,multiple_collectives/all_gather_size_134217728...,1,NS3,DCQCN2_k3200-800_pkt9000,109054799,6.904998e+06,727.832189,16,/app/astra-sim/upc/output/comparison_run/exper...
2,multiple_collectives/all_gather_size_134217728...,1,NS3,HPCC_pkt9000,122688969,8.778221e+06,539.839433,16,/app/astra-sim/upc/output/comparison_run/exper...
...,...,...,...,...,...,...,...,...,...
7,multiple_collectives/all_gather_size_33554432_...,-1,Analytical Unaware,Analytical Unaware,1342197,0.000000e+00,1.192587,16,/app/astra-sim/upc/output/comparison_run/exper...
6,multiple_collectives/all_gather_size_33554432_...,1,G2,G2,25603413,1.857253e+06,1.314317,16,/app/astra-sim/upc/output/comparison_run/exper...
8,multiple_collectives/all_gather_size_33554432_...,1,NS3,DCTCP_pkt9000,25934623,1.144910e+06,128.100193,16,/app/astra-sim/upc/output/comparison_run/exper...
9,multiple_collectives/all_gather_size_33554432_...,1,NS3,HPCC_pkt9000,30809423,2.163307e+06,150.201511,16,/app/astra-sim/upc/output/comparison_run/exper...
